In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)
X = 2 * np.random.rand(100, 1) 
y = 4 + 3 * X.flatten() + np.random.randn(100)

print("Feature values (X):", X[:10].flatten())
print("Target values (y):", y[:10])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

def manual_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def manual_mse(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

def manual_rmse(y_true, y_pred):
    return np.sqrt(manual_mse(y_true, y_pred))

def manual_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    return 1 - (ss_res / ss_tot)

results = {
    "Metric": ["MAE", "MSE", "RMSE", "R2"],
    "Manual": [manual_mae(y_test, y_pred), manual_mse(y_test, y_pred), 
               manual_rmse(y_test, y_pred), manual_r2(y_test, y_pred)],
    "Built-in": [mean_absolute_error(y_test, y_pred), mean_squared_error(y_test, y_pred), 
                 np.sqrt(mean_squared_error(y_test, y_pred)), r2_score(y_test, y_pred)]
}
print("\nRegression Metrics Verification:")
print(pd.DataFrame(results))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv('cancer.csv')

df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

X_class = df.drop(columns=['id', 'diagnosis'], errors='ignore')
y_class = df['diagnosis']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_class, y_class, test_size=0.2, random_state=42)

clf = LogisticRegression(max_iter=10000)
clf.fit(X_train_c, y_train_c)
y_pred_c = clf.predict(X_test_c)

def manual_confusion_matrix(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return np.array([[tn, fp], [fn, tp]]), tp, tn, fp, fn

cm_custom, TP, TN, FP, FN = manual_confusion_matrix(y_test_c.values, y_pred_c)

m_acc = (TP + TN) / (TP + TN + FP + FN)
m_pre = TP / (TP + FP) if (TP + FP) > 0 else 0
m_rec = TP / (TP + FN) if (TP + FN) > 0 else 0
m_f1 = 2 * (m_pre * m_rec) / (m_pre + m_rec) if (m_pre + m_rec) > 0 else 0

print("\nClassification Metrics Verification:")
print(f"{'Metric':<12} | {'Manual':<10} | {'Built-in':<10}")
print("-" * 38)
print(f"{'Accuracy':<12} | {m_acc:<10.4f} | {accuracy_score(y_test_c, y_pred_c):<10.4f}")
print(f"{'Precision':<12} | {m_pre:<10.4f} | {precision_score(y_test_c, y_pred_c):<10.4f}")
print(f"{'Recall':<12} | {m_rec:<10.4f} | {recall_score(y_test_c, y_pred_c):<10.4f}")
print(f"{'F1-Score':<12} | {m_f1:<10.4f} | {f1_score(y_test_c, y_pred_c):<10.4f}")

print("\nConfusion Matrix (Manual):\n", cm_custom)
print("Confusion Matrix (Built-in):\n", confusion_matrix(y_test_c, y_pred_c))